In [ ]:

import time
import cv2
import numpy as np
import os
from sdlarch_rl.utils.utils import get_latest_model, FrameSkip, AugmentObservation
from stable_baselines3.common.atari_wrappers import WarpFrame
from stable_baselines3.common.env_util import make_vec_env
from stable_baselines3.common.vec_env import DummyVecEnv, VecFrameStack
from pathlib import Path
from sdlarch_rl.utils.utils import get_last_index, RealExcludeButtonsWrapper, GenericCNN, TimeLimit
from sdlarch_rl import make
from stable_baselines3.common.policies import ActorCriticPolicy
from stable_baselines3 import PPO, A2C
from gymnasium.wrappers.frame_stack import FrameStack
import gymnasium as gym

DETERMINISTIC=False

train_path = 'imitation-gt3/'

last_index_imitation = int(get_last_index(train_path, "bc_policy", "zip"))
latest_model_path = train_path + f"bc_policy{last_index_imitation}.zip"

print("loading from: " + str(latest_model_path))

class ResetOnSpeedStateWrapper(gym.Wrapper):
    def __init__(self, env):
        super().__init__(env)
        
        self.env = env

        self.low_velociy_count = 0

    def reset(self, **kwargs):

        self.low_velociy_count = 0

        return self.env.reset(**kwargs)

    def step(self, action):
        obs, rew, done, trunc, info = self.env.step(action)

        # print(info)

        velocity = info.get("velocity", 0)

        if velocity < 20:
            self.low_velociy_count += 1

            if self.low_velociy_count > 600:
                done = True
                rew = 1.0
        else:
            self.low_velociy_count = 0

        return obs, rew, done, trunc, info

global myEnv

def make_env():
    global myEnv
    myEnv = make(
        "GranTurismo3-Ps2", 
        statename="middle_field",
        render_mode="human"
    )
    
    buttons = myEnv.unwrapped.buttons
    to_exclude = ["UP", "DOWN", "START", "SELECT", "R1", "L1", "L2", "R2", "L3", "R3", "A"]

    env = ResetOnSpeedStateWrapper(myEnv)
    env = RealExcludeButtonsWrapper(env, buttons, to_exclude)
    env = WarpFrame(env, width=96, height=96)
    env = FrameSkip(env, skip=4)
    env = TimeLimit(env, max_steps=4000)

    return env

env = make_env()
env = FrameStack(env, 4)

print(env.action_space)

a2c = ActorCriticPolicy.load(
    str(latest_model_path),
)

policy_kwargs = dict(
    net_arch=dict(pi=[256, 256], vf=[256, 256]), 
    features_extractor_class=GenericCNN
)

ppo = PPO(
    policy=a2c.__class__,
    env=env,
    policy_kwargs=policy_kwargs
)

ppo.policy.load_state_dict(a2c.state_dict(), strict=False)


SCREEN_WIDTH = 640*3
SCREEN_HEIGHT = 480*3

prev_keys = set()

obs, _ = env.reset()

color=(0, 0, 255)

while True:
    # uncomment if ActorCriticPolicy
    obs = np.array(obs)
    # if len(obs.shape) == 4 and obs.shape[3] == 1:
    #     obs = np.array(obs).squeeze(-1)
    
    action, _ = ppo.predict(obs, deterministic=DETERMINISTIC)
    # action, _ = a2c.predict(obs, deterministic=DETERMINISTIC)

    obs, rew, done, trunc, _ = env.step(action)

    img = myEnv.render()

    if done or trunc:
        env.reset()

    if img is not None:
        img = cv2.resize(img, (SCREEN_WIDTH, SCREEN_HEIGHT))
        img = cv2.cvtColor(img, cv2.COLOR_RGB2BGR)
        
        cv2.circle(img, center=(100, 100), radius=50, color=color, thickness=2)
        
        cv2.imshow("game", img)
    
        cv2.waitKey(1)

# s.close()

D:\Python311\Lib\site-packages\pygame\pkgdata.py:25: DeprecationWarning: pkg_resources is deprecated as an API. See https://setuptools.pypa.io/en/latest/pkg_resources.html
  from pkg_resources import resource_stream, resource_exists


loading from: imitation-gt3/bc_policy11.zip
Pygame initialized: 640x448
MultiBinary(5)


D:\Python311\Lib\site-packages\stable_baselines3\common\policies.py:176: FutureWarning: You are using `torch.load` with `weights_only=False` (the current default value), which uses the default pickle module implicitly. It is possible to construct malicious pickle data which will execute arbitrary code during unpickling (See https://github.com/pytorch/pytorch/blob/main/SECURITY.md#untrusted-models for more details). In a future release, the default value for `weights_only` will be flipped to `True`. This limits the functions that could be executed during unpickling. Arbitrary objects will no longer be allowed to be loaded via this mode unless they are explicitly allowlisted by the user via `torch.serialization.add_safe_globals`. We recommend you start setting `weights_only=True` for any use case where you don't have full control of the loaded file. Please open an issue on GitHub for any issues related to this experimental feature.
  saved_variables = th.load(path, map_location=device)
